# IDVerse API — Interactive Notebook

Mirrors the Bruno collection. Set `ENV` in the config cell and run from top to bottom.

**Sections:** Auth · Verification · Status · Webhook · OAuth Test · Mock · Upstream IDVerse

In [5]:
# ============================================================
# CONFIGURATION — run this cell first, then any section below
# ============================================================
ENV = "local"  # Options: "local", "dev", "prod"

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs

env_file = f".env.{ENV}"
if not Path(env_file).exists():
    print(f"⚠️  {env_file} not found — copy .env.example to {env_file} and fill in values")
else:
    load_dotenv(env_file, override=True)
    print(f"Loaded {env_file}")

BASE_URL              = os.getenv("BASE_URL", "http://localhost:19746")
AUTH_KEY              = os.getenv("AUTH_KEY", "")
IDVERSE_CLIENT_ID     = os.getenv("IDVERSE_CLIENT_ID", "")
IDVERSE_CLIENT_SECRET = os.getenv("IDVERSE_CLIENT_SECRET", "")
IDVERSE_OAUTH_URL     = os.getenv("IDVERSE_OAUTH_URL", "https://us.demo.idkit.co/api/3.5/oauthToken")
IDVERSE_API_URL       = os.getenv("IDVERSE_API_URL", "https://us.demo.idkit.co/api/3.5/sendSms")
IDVERSE_ACCESS_TOKEN  = os.getenv("IDVERSE_ACCESS_TOKEN", "")
WEBHOOK_JWT_TOKEN     = os.getenv("WEBHOOK_JWT_TOKEN", "")
NOTIFY_URL_COMPLETE   = os.getenv("NOTIFY_URL_COMPLETE", f"{BASE_URL}/api/webhook")
NOTIFY_URL_EVENT      = os.getenv("NOTIFY_URL_EVENT", f"{BASE_URL}/api/webhook")

# Sample request data — edit as needed
PHONE_CODE     = "+1"
PHONE_NUMBER   = "9547017082"
REFERENCE_ID   = "test-ref-001"
TRANSACTION_ID = "txn-test-001"

def ok(val):
    return "set" if val else "NOT SET"

print(f"Environment  : {ENV}")
print(f"Base URL     : {BASE_URL}")
print(f"AUTH_KEY     : {ok(AUTH_KEY)}")
print(f"CLIENT_ID    : {ok(IDVERSE_CLIENT_ID)}")
print(f"WEBHOOK JWT  : {ok(WEBHOOK_JWT_TOKEN)}")
print(f"ACCESS TOKEN : {ok(IDVERSE_ACCESS_TOKEN)}")

Loaded .env.local
Environment  : local
Base URL     : http://localhost:19746
AUTH_KEY     : set
CLIENT_ID    : set
WEBHOOK JWT  : NOT SET
ACCESS TOKEN : NOT SET


---
## Auth

In [6]:
# GET /api/getAuth
# Returns a 302 redirect to /?jwt_key=<token> on success.
resp = requests.get(
    f"{BASE_URL}/api/getAuth",
    params={"auth_key": AUTH_KEY},
    allow_redirects=False
)
print(f"Status: {resp.status_code}")

if resp.status_code == 302:
    location = resp.headers.get("Location", "")
    qs = parse_qs(urlparse(location).query)
    jwt_key = qs.get("jwt_key", [""])[0]
    print(f"Redirect: {location}")
    preview = jwt_key[:40] + "..." if len(jwt_key) > 40 else jwt_key
    print(f"jwt_key : {preview}")
    assert jwt_key, "Expected jwt_key in redirect URL"
    print("✓ Auth succeeded")
elif resp.status_code == 403:
    print("403 Forbidden — check AUTH_KEY in your .env file")
    print(resp.json())
else:
    print(resp.text[:300])

Status: 302
Redirect: http://localhost:19746/?jwt_key=aHjOR
jwt_key : aHjOR
✓ Auth succeeded


---
## Verification

In [7]:
# POST /api/verify — real verification, sends SMS via IDVerse API
payload = {
    "phoneCode": PHONE_CODE,
    "phoneNumber": PHONE_NUMBER,
    "referenceId": REFERENCE_ID,
    "transactionId": TRANSACTION_ID,
    "name": "John Doe",
    "suppliedFirstName": "John"
}
resp = requests.post(f"{BASE_URL}/api/verify", json=payload)
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert "status" in data, "Response missing 'status' field"
print(f"\n✓ status = {data['status']}")

Status: 400
{
  "message": "Access denied by CloudFront. The request was blocked - please check IP restrictions or contact IDVerse support.",
  "status": "error"
}

✓ status = error


In [8]:
# POST /api/verify/test?dryRun=true — no SMS sent, saves mock record
payload = {
    "phoneCode": PHONE_CODE,
    "phoneNumber": PHONE_NUMBER,
    "referenceId": REFERENCE_ID,
    "transactionId": TRANSACTION_ID,
    "name": "John Doe",
    "suppliedFirstName": "John"
}
resp = requests.post(f"{BASE_URL}/api/verify/test", json=payload, params={"dryRun": "true"})
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
assert data.get("dryRun") == "true", "Expected dryRun=true in response"
assert "transactionId" in data, "Response missing transactionId"
print(f"\n✓ dry run ok — transactionId = {data['transactionId']}")

Status: 200
{
  "dryRun": "true",
  "transactionId": "txn-test-001",
  "status": "success"
}

✓ dry run ok — transactionId = txn-test-001


In [9]:
# POST /api/verify/test?dryRun=false — real call through test endpoint (no auth required)
payload = {
    "phoneCode": PHONE_CODE,
    "phoneNumber": PHONE_NUMBER,
    "referenceId": REFERENCE_ID,
    "transactionId": TRANSACTION_ID,
    "name": "John Doe",
    "suppliedFirstName": "John"
}
resp = requests.post(f"{BASE_URL}/api/verify/test", json=payload, params={"dryRun": "false"})
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert "status" in data, "Response missing 'status' field"
assert data.get("dryRun") == "false", "Expected dryRun=false in response"
print(f"\n✓ status = {data['status']}")

Status: 400
{
  "dryRun": "false",
  "message": "Access denied by CloudFront. The request was blocked - please check IP restrictions or contact IDVerse support.",
  "status": "error"
}

✓ status = error


In [10]:
# GET /api/verifications — all records
resp = requests.get(f"{BASE_URL}/api/verifications")
print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Records: {len(data)}")
if data:
    print("Latest record:")
    print(json.dumps(data[-1], indent=2))

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
assert isinstance(data, list), "Expected list response"
print(f"\n✓ {len(data)} verification record(s)")

Status: 200
Records: 4
Latest record:
{
  "id": 4,
  "phoneNumber": "+19547017082",
  "referenceId": "test-ref-001",
  "transactionId": "txn-test-001",
  "apiResponse": null,
  "status": "FAILURE",
  "timestamp": "2026-05-08T11:39:17.348681",
  "errorMessage": "Access denied by CloudFront. The request was blocked - please check IP restrictions or contact IDVerse support."
}

✓ 4 verification record(s)


In [11]:
# GET /api/verifications/{id} — single record by database ID
record_id = 1  # change to a valid ID
resp = requests.get(f"{BASE_URL}/api/verifications/{record_id}")
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    print(json.dumps(resp.json(), indent=2))
    assert "id" in resp.json(), "Response missing 'id' field"
    print(f"\n✓ found record {record_id}")
elif resp.status_code == 404:
    print(f"Record {record_id} not found — change record_id above")
else:
    print(resp.text)

Status: 200
{
  "id": 1,
  "phoneNumber": "+15551234567",
  "referenceId": "test-ref-001",
  "transactionId": "txn-test-001",
  "apiResponse": null,
  "status": "FAILURE",
  "timestamp": "2026-05-08T11:38:04.292456",
  "errorMessage": "Access denied by CloudFront. The request was blocked - please check IP restrictions or contact IDVerse support."
}

✓ found record 1


---
## Status

In [ ]:
# GET /api/status/reference/{referenceId} — latest status for a reference ID
ref_id = REFERENCE_ID  # change as needed
resp = requests.get(f"{BASE_URL}/api/status/reference/{ref_id}")
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    print(json.dumps(data, indent=2))
    assert "status" in data, "Response missing 'status' field"
    assert "timestamp" in data, "Response missing 'timestamp' field"
    print(f"\n✓ status = {data['status']}")
elif resp.status_code == 404:
    print(f"No record found for referenceId={ref_id}")
else:
    print(resp.text)

In [ ]:
# GET /api/status/transaction/{transactionId} — latest status for a transaction ID
txn_id = TRANSACTION_ID  # change as needed
resp = requests.get(f"{BASE_URL}/api/status/transaction/{txn_id}")
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    print(json.dumps(data, indent=2))
    assert "status" in data, "Response missing 'status' field"
    assert "timestamp" in data, "Response missing 'timestamp' field"
    print(f"\n✓ status = {data['status']}")
elif resp.status_code == 404:
    print(f"No record found for transactionId={txn_id}")
else:
    print(resp.text)

---
## Webhook

Requires `WEBHOOK_JWT_TOKEN` to be set. See README for how to get it.

Valid event values: `pending`, `termsAndConditions`, `idSelection`, `personalDetails`, `liveness`, `expired`, `cancelled`, `completedPass`, `completedFlagged`

In [ ]:
# POST /api/webhook — simulate completion webhook (completedPass)
if not WEBHOOK_JWT_TOKEN:
    print("⚠️  WEBHOOK_JWT_TOKEN not set — skipping")
else:
    payload = {
        "transactionId": TRANSACTION_ID,
        "event": "completedPass"
    }
    resp = requests.post(
        f"{BASE_URL}/api/webhook",
        json=payload,
        headers={"Authorization": f"Bearer {WEBHOOK_JWT_TOKEN}"}
    )
    print(f"Status: {resp.status_code}")
    data = resp.json()
    print(json.dumps(data, indent=2))

    assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
    assert data.get("status") == "success", f"Expected success, got {data.get('status')}"
    print(f"\n✓ webhook processed, recordId = {data.get('recordId')}")

In [ ]:
# POST /api/webhook — simulate event webhook (pending)
# Change 'event' to any valid value to test different states
if not WEBHOOK_JWT_TOKEN:
    print("⚠️  WEBHOOK_JWT_TOKEN not set — skipping")
else:
    payload = {
        "transactionId": TRANSACTION_ID,
        "event": "pending"  # try: termsAndConditions, idSelection, liveness, expired, cancelled
    }
    resp = requests.post(
        f"{BASE_URL}/api/webhook",
        json=payload,
        headers={"Authorization": f"Bearer {WEBHOOK_JWT_TOKEN}"}
    )
    print(f"Status: {resp.status_code}")
    data = resp.json()
    print(json.dumps(data, indent=2))

    assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
    assert data.get("status") == "success"
    print(f"\n✓ event webhook processed, recordId = {data.get('recordId')}")

In [ ]:
# POST /api/updateStatus — same as webhook but no JWT required
# Use 'event' (camelCase IDVerse name) OR 'status' (explicit uppercase string); event takes precedence
payload = {
    "transactionId": TRANSACTION_ID,
    "event": "completedPass"  # or use "status": "COMPLETED PASS"
}
resp = requests.post(f"{BASE_URL}/api/updateStatus", json=payload)
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
assert data.get("status") == "success"
assert "resolvedStatus" in data, "Response missing 'resolvedStatus'"
print(f"\n✓ resolvedStatus = {data['resolvedStatus']}")

---
## OAuth Test

In [ ]:
# GET /test/oauth — test OAuth token retrieval
resp = requests.get(f"{BASE_URL}/test/oauth")
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert "status" in data, "Response missing 'status' field"
if data["status"] == "SUCCESS":
    print(f"\n✓ OAuth ok — token_type={data.get('token_type')}, expires_in={data.get('expires_in')}")
else:
    print(f"\n✗ OAuth failed — check IDVERSE_CLIENT_ID and IDVERSE_CLIENT_SECRET")

In [ ]:
# GET /test/oauth?verbose=debug — full request/response detail
resp = requests.get(f"{BASE_URL}/test/oauth", params={"verbose": "debug"})
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert "status" in data

In [ ]:
# POST /test/oauth/clear — clear the 800-second token cache
resp = requests.post(f"{BASE_URL}/test/oauth/clear")
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert resp.status_code == 200
assert data.get("status") == "SUCCESS"
print("\n✓ token cache cleared")

In [ ]:
# GET /test/config — validates app configuration end-to-end
resp = requests.get(f"{BASE_URL}/test/config")
print(f"Status: {resp.status_code}")
data = resp.json()
print(json.dumps(data, indent=2))

assert resp.status_code == 200
assert "status" in data
print(f"\n✓ config status = {data['status']}")

---
## Mock

In [ ]:
# POST /api/3.5/oauthToken — local mock OAuth endpoint
# Returns the OAUTHTOKEN value from the app's .env
resp = requests.post(f"{BASE_URL}/api/3.5/oauthToken")
print(f"Status: {resp.status_code}")
data = resp.json()
# Mask the token in output
if "accessToken" in data and data["accessToken"]:
    display = dict(data)
    display["accessToken"] = data["accessToken"][:20] + "..."
    print(json.dumps(display, indent=2))
else:
    print(json.dumps(data, indent=2))

assert resp.status_code in [200, 500], f"Unexpected status: {resp.status_code}"
if resp.status_code == 200:
    print(f"\n✓ mock token returned, type={data.get('tokenType')}")
else:
    print("\n⚠️  OAUTHTOKEN not set in app .env — expected if not configured")

---
## Upstream IDVerse

Direct calls to the IDVerse API, bypassing the local app.

1. Run **Get OAuth Token** — copy `access_token` into `IDVERSE_ACCESS_TOKEN` in your `.env` file and re-run the config cell
2. Run **Send SMS Verification**

In [ ]:
# POST to IDVERSE_OAUTH_URL — get an OAuth token directly from IDVerse
if not IDVERSE_CLIENT_ID or not IDVERSE_CLIENT_SECRET:
    print("⚠️  IDVERSE_CLIENT_ID or IDVERSE_CLIENT_SECRET not set — skipping")
else:
    resp = requests.post(
        IDVERSE_OAUTH_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": IDVERSE_CLIENT_ID,
            "client_secret": IDVERSE_CLIENT_SECRET
        }
    )
    print(f"Status: {resp.status_code}")
    data = resp.json()

    # Mask token in output
    display = dict(data)
    if "access_token" in display and display["access_token"]:
        preview = display["access_token"][:20] + "..."
        print(json.dumps({**display, "access_token": preview}, indent=2))
        print(f"\n✓ token obtained — paste the full access_token into IDVERSE_ACCESS_TOKEN in .env.{ENV}")
    else:
        print(json.dumps(display, indent=2))
        print("\n✗ no access_token in response")

    assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"

In [ ]:
# POST to IDVERSE_API_URL — send SMS verification directly to IDVerse
# Requires IDVERSE_ACCESS_TOKEN from the cell above
if not IDVERSE_ACCESS_TOKEN:
    print("⚠️  IDVERSE_ACCESS_TOKEN not set — run Get OAuth Token first")
else:
    payload = {
        "phoneCode": PHONE_CODE,
        "phoneNumber": PHONE_NUMBER,
        "referenceId": REFERENCE_ID,
        "transactionId": TRANSACTION_ID,
        "name": "John Doe",
        "suppliedFirstName": "John",
        "notifyUrlComplete": NOTIFY_URL_COMPLETE,
        "notifyUrlCompleteAuthKey": f"Bearer {WEBHOOK_JWT_TOKEN}",
        "notifyUrlCompleteAuthHeaderName": "Authorization",
        "notifyUrlEvent": NOTIFY_URL_EVENT,
        "notifyUrlEventAuthKey": f"Bearer {WEBHOOK_JWT_TOKEN}",
        "notifyUrlEventAuthHeaderName": "Authorization"
    }
    resp = requests.post(
        IDVERSE_API_URL,
        json=payload,
        headers={
            "Authorization": f"Bearer {IDVERSE_ACCESS_TOKEN}",
            "accept": "application/json"
        }
    )
    print(f"Status: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text[:500])

    assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text[:200]}"